# Knapsack Benchmark — Analysis and Plots

Publication-ready analysis notebook for the benchmark results. Plots are saved as 300 DPI PNGs in `results/plots/`.

In [3]:
# Section 0: Imports and global plotting style
from __future__ import annotations

import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy.optimize import curve_fit
except Exception:
    curve_fit = None

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

def resolve_root_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'results').exists():
        return cwd
    if (cwd.parent / 'results').exists():
        return cwd.parent
    return cwd

ROOT_DIR = resolve_root_dir()
PLOTS_DIR = ROOT_DIR / 'results' / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig: plt.Figure, name: str) -> None:
    out_path = PLOTS_DIR / name
    fig.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches='tight')

print('Root dir:', ROOT_DIR)

Root dir: E:\Antigravity Workspace\Knapsack\KnapsackOptimization


## Section 1: Robust Data Preprocessing

In [ ]:
CSV_PATH = ROOT_DIR / "results" / "csv" / "benchmark_results_timeout5.csv"
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

df = pd.DataFrame(df)

# Normalize numeric columns defensively
for col in ["time_sec", "peak_memory_mb", "optimal_value", "n", "capacity", "pearson_corr", "density_variance"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Timeout handling
TIMEOUT_SEC = int(os.getenv("BENCHMARK_TIMEOUT_SEC", "60"))
df["is_timeout"] = df["status"].str.upper().eq("TIMEOUT")
df.loc[df["is_timeout"], "time_sec"] = TIMEOUT_SEC

# OOM handling
df["is_oom"] = df["status"].str.upper().eq("OOM")
max_mem = df["peak_memory_mb"].max(skipna=True) if "peak_memory_mb" in df.columns else np.nan
default_oom = 32768.0  # predefined massive outlier (MB)
OOM_MEMORY_MB = float(max_mem * 2) if pd.notna(max_mem) and max_mem > 0 else default_oom
df.loc[df["is_oom"], "peak_memory_mb"] = OOM_MEMORY_MB

# Errors/timeout memory should not be plotted as zero
error_mask = df["status"].str.upper().isin(["TIMEOUT", "ERROR"])
df.loc[error_mask, "peak_memory_mb"] = np.nan


def shorten_test_id(value: str) -> str:
    # Extract compact suffix like n100_cr0.5 if present
    parts = str(value).split("_")
    compact = [p for p in parts if p.startswith("n") or p.startswith("cr")]
    return "_".join(compact) if compact else str(value)


# Basic diagnostics
print("Rows:", len(df))
print("Algorithms:", df["algorithm"].nunique())
print("Timeouts:", df["is_timeout"].sum(), "OOM:", df["is_oom"].sum())

In [ ]:
## Section 2: Time Complexity and Empirical Curve Fitting
#
# Focus on near-uncorrelated data to minimize structural bias
#
df_uncorr = df.copy()
df_uncorr = df_uncorr[pd.notna(df_uncorr['pearson_corr'])]
df_uncorr = df_uncorr[df_uncorr['pearson_corr'].abs() <= 0.1]
df_uncorr = df_uncorr[df_uncorr['time_sec'] > 0]

# Remove failed rows for fitting; retain TIMEOUT (capped) for scatter
df_fit = df_uncorr[df_uncorr["status"].str.upper().eq("SUCCESS")].copy()

def exp_func(x, a, b):
    return a * (2 ** (b * x))

def linear_func(x, a, b):
    return a * x + b

fig, ax = plt.subplots(figsize=(7.5, 4.5))

# Main scatter for successful runs
sns.scatterplot(
    data=df_uncorr[df_uncorr["status"].str.upper().eq("SUCCESS")],
    x="n",
    y="time_sec",
    hue="algorithm",
    alpha=0.7,
    ax=ax,
)

# Mark timeouts/errors as red X
fail_mask = df_uncorr["status"].str.upper().isin(["TIMEOUT", "ERROR"])
if fail_mask.any():
    ax.scatter(
        df_uncorr.loc[fail_mask, "n"],
        df_uncorr.loc[fail_mask, "time_sec"],
        marker="x",
        s=40,
        color="red",
        label="TIMEOUT/ERROR",
    )

ax.set_yscale("log")
ax.set_title("Empirical Runtime vs n (uncorrelated instances)")
ax.set_xlabel("n (number of items)")
ax.set_ylabel("time_sec (log scale)")

# Curve fitting for brute-force style (if present)
if curve_fit is None:
    print("scipy not available; skip curve fitting.")
else:
    brute_candidates = [
        name
        for name in df_fit["algorithm"].unique()
        if "Brute" in name or "Backtracking" in name
    ]
    for algo in brute_candidates:
        sub = df_fit[df_fit["algorithm"] == algo]
        if len(sub) < 3:
            continue
        x = sub["n"].values.astype(float)
        y = sub["time_sec"].values.astype(float)
        try:
            params, _ = curve_fit(exp_func, x, y, p0=(1e-6, 0.5), maxfev=10000)
            x_grid = np.linspace(x.min(), x.max(), 100)
            y_fit = exp_func(x_grid, *params)
            ax.plot(x_grid, y_fit, linestyle="--", label=f"{algo} fit: a*2^(b*n)")
        except Exception as exc:
            print(f"Fit failed for {algo}:", exc)

    # Curve fitting for DP-style (pseudo-polynomial)
    dp_candidates = [name for name in df_fit["algorithm"].unique() if name.startswith("DP")]
    for algo in dp_candidates:
        sub = df_fit[(df_fit["algorithm"] == algo) & (df_fit["knapsack_type"] == "01")]
        if len(sub) < 3:
            continue
        x2 = (sub["n"] * sub["capacity"]).astype(float).values
        y = sub["time_sec"].values.astype(float)
        try:
            params, _ = curve_fit(linear_func, x2, y, p0=(1e-9, 0.0), maxfev=10000)
            n_unique = np.sort(sub["n"].unique())
            median_capacity = sub.groupby("n")["capacity"].median().reindex(n_unique).values
            x2_proj = n_unique * median_capacity
            y_fit = linear_func(x2_proj, *params)
            ax.plot(n_unique, y_fit, linestyle="--", label=f"{algo} fit: a*(n*W)+b")
        except Exception as exc:
            print(f"Fit failed for {algo}:", exc)

ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
save_fig(fig, "time_complexity_curve_fit.png")
plt.show()

In [ ]:
## Section 3: The Space Complexity Curse of DP (Memory Profiling)

In [ ]:
# Prefer memory stress scenarios if available; fallback to high-capacity subset
stress_mask = df['test_id'].astype(str).str.contains('stress', case=False, na=False)
df_stress = df[stress_mask].copy()

if df_stress.empty:
    # Fallback: pick the most common n and top 10% capacities
    if not df.empty:
        common_n = df['n'].mode().iloc[0]
        cap_threshold = df['capacity'].quantile(0.9)
        df_stress = df[(df['n'] == common_n) & (df['capacity'] >= cap_threshold)].copy()

if df_stress.empty:
    print("No memory stress data found; skipping Section 3 plot.")
else:
    df_stress = df_stress[df_stress["status"].str.upper().isin(["SUCCESS", "OOM", "TIMEOUT"])]
    df_stress["capacity_str"] = df_stress["capacity"].astype(int)

    def _format_capacity(value: int) -> str:
        if value >= 1_000_000:
            return f"{value/1_000_000:.1f}M"
        if value >= 1_000:
            return f"{value/1_000:.1f}k"
        return str(value)

    df_stress["capacity_label"] = df_stress["capacity_str"].map(_format_capacity)

    fig, ax = plt.subplots(figsize=(8.0, 4.5))
    sns.barplot(
        data=df_stress,
        x="capacity_label",
        y="peak_memory_mb",
        hue="algorithm",
        ax=ax,
    )
    ax.set_title("Memory Footprint vs Capacity (stress scenarios)")
    ax.set_xlabel("Capacity")
    ax.set_ylabel("Peak Memory (MB)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

    # Annotate OOM bars
    x_order = [t.get_text() for t in ax.get_xticklabels()]
    legend = ax.get_legend()
    hue_order = [t.get_text() for t in legend.texts] if legend else []

    for hue_idx, container in enumerate(ax.containers):
        algo = hue_order[hue_idx] if hue_idx < len(hue_order) else None
        for cap_idx, bar in enumerate(container):
            if algo is None or cap_idx >= len(x_order):
                continue
            cap = x_order[cap_idx]
            is_oom = (
                (df_stress["algorithm"] == algo)
                & (df_stress["capacity_label"] == cap)
                & (df_stress["is_oom"])
            ).any()
            if not is_oom:
                continue
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01,
                "OOM",
                ha="center",
                va="bottom",
                fontsize=8,
                color="crimson",
            )

    # Mark timeouts as dashed vertical lines
    timeout_caps = (
        df_stress[df_stress["status"].str.upper().eq("TIMEOUT")]["capacity_label"].unique()
    )
    for cap in timeout_caps:
        if cap in x_order:
            idx = x_order.index(cap)
            ax.axvline(idx, color="#888888", linestyle="--", linewidth=1)

    save_fig(fig, "memory_stress_dp_vs_capacity.png")
    plt.show()

In [ ]:
## Section 4: Data Correlation vs Pruning Efficiency in Branch and Bound

In [ ]:
# Prefer the standard BranchAndBound if present; otherwise any algorithm with BranchAndBound in name
if (df['algorithm'] == 'BranchAndBound').any():
    bnb_df = df[df['algorithm'] == 'BranchAndBound'].copy()
else:
    bnb_df = df[df['algorithm'].str.contains('BranchAndBound', na=False)].copy()

bnb_df = bnb_df[bnb_df['status'].str.upper().isin(['SUCCESS', 'TIMEOUT'])]

if bnb_df.empty:
    print('No Branch and Bound data found; skipping Section 4 plot.')
else:
    fig, ax = plt.subplots(figsize=(6.5, 4.0))
    sns.regplot(
        data=bnb_df,
        x="pearson_corr",
        y="time_sec",
        x_jitter=0.05,
        scatter_kws={"alpha": 0.6},
        line_kws={"color": "black"},
        ax=ax,
    )
    ax.set_title("Pearson Correlation vs Branch and Bound Runtime")
    ax.set_xlabel("Pearson correlation (value vs weight)")
    ax.set_ylabel("time_sec")
    save_fig(fig, "bnb_pearson_vs_time.png")
    plt.show()

In [ ]:
## Section 5: The Knapsack Hierarchy (Fractional vs Unbounded vs 0/1)

In [ ]:
success_df = df[df['status'].str.upper().eq('SUCCESS')].copy()
types_needed = {'fractional', 'unbounded', '01'}

# Prefer exact test_id matches; fallback to (n, capacity) if needed
by_test = success_df.groupby('test_id')['knapsack_type'].apply(set)
valid_test_ids = by_test[by_test.apply(lambda s: types_needed.issubset(s))].index

if len(valid_test_ids) > 0:
    aligned = success_df[success_df['test_id'].isin(valid_test_ids)].copy()
    group_cols = ['test_id', 'knapsack_type']
else:
    success_df['key'] = list(zip(success_df['n'], success_df['capacity']))
    by_key = success_df.groupby('key')['knapsack_type'].apply(set)
    valid_keys = by_key[by_key.apply(lambda s: types_needed.issubset(s))].index
    aligned = success_df[success_df['key'].isin(valid_keys)].copy()
    group_cols = ['key', 'knapsack_type']

if aligned.empty:
    print('No aligned instances found for hierarchy plot.')
else:
    # Aggregate: best value and best (min) time per knapsack type in each aligned group
    agg = aligned.groupby(group_cols).agg(
        optimal_value=('optimal_value', 'max'),
        time_sec=('time_sec', 'min'),
    ).reset_index()
    summary = agg.groupby('knapsack_type').agg(
        mean_optimal_value=('optimal_value', 'mean'),
        mean_time_sec=('time_sec', 'mean'),
    ).reindex(['fractional', 'unbounded', '01'])

    fig, ax1 = plt.subplots(figsize=(6.5, 4.0))
    ax2 = ax1.twinx()

    x = np.arange(len(summary.index))
    bars = ax1.bar(x, summary["mean_optimal_value"], color="steelblue", alpha=0.8)
    ax2.plot(x, summary["mean_time_sec"], color="darkred", marker="o", label="Mean time")

    ax1.set_xticks(x)
    ax1.set_xticklabels(summary.index)
    ax1.set_ylabel("Mean Optimal Value")
    ax2.set_ylabel("Mean Time (sec)")
    ax1.set_title("Knapsack Hierarchy: Value vs Time")
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha="right")

    # Annotate bars
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, height, f"{height:.1f}", ha="center", va="bottom", fontsize=8)

    ax2.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0)
    save_fig(fig, "knapsack_hierarchy_value_time.png")
    plt.show()